In [4]:
from torchvision import datasets
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score
DATA_DIR = "./fashion_mnist_data"
DOWNLOAD = True

raw_train = datasets.FashionMNIST(
    DATA_DIR,
    train=True,
    download=DOWNLOAD
)

raw_val = datasets.FashionMNIST(
    DATA_DIR,
    train=False,
    download=DOWNLOAD
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device  : {device}")
print(f"Train samples : {len(raw_train)}")
print(f"Val samples   : {len(raw_val)}")
print("Fashion-MNIST ready — run cell 2 to configure and train")

Using device  : cuda
Train samples : 60000
Val samples   : 10000
Fashion-MNIST ready — run cell 2 to configure and train


In [25]:
# ─── CV CONFIG ────────────────────────────────────────────────────────────────
CV_CONFIG = {
    "image_size"    : 128,
    "in_channels"   : 1,
    "num_classes"   : 10,
    "row_col_channels" : 8,        # output channels for RowExtractor & ColExtractor
    "conv2d_layers" : [
        # (out_ch, kh, kw, padding_w)   stride is always 1, maxpool after block if True
        # (out_ch, kh, kw, pad_w, maxpool)
        (16,  2, 3, 1, False),   # Block1: collapses height 2→1, no pool
        (32,  1, 3, 1, True),    # Block2: + MaxPool(1,2)
        (64,  1, 3, 1, True),    # Block3: + MaxPool(1,2)
        (128, 1, 3, 1, True),    # Block4: + MaxPool(1,2)
    ],
    "fc_layers"     : [64, 32],
    "dropout"       : 0.1,
    "batch_size"    : 64,
    "epochs"        : 20,
    "lr"            : 1e-3,
    "n_folds"       : 5,
}
# ──────────────────────────────────────────────────────────────────────────────


class RowExtractor(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        img_size  = cfg["image_size"]
        in_ch     = cfg["in_channels"]
        out_ch    = cfg["row_col_channels"]
        self.conv = nn.Conv2d(in_ch, out_ch,
                              kernel_size=(1, img_size),
                              stride=1, padding=0)

    def forward(self, x):
        return self.conv(x).squeeze(-1)   # (B, row_col_channels, 128)


class ColExtractor(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        img_size  = cfg["image_size"]
        in_ch     = cfg["in_channels"]
        out_ch    = cfg["row_col_channels"]
        self.conv = nn.Conv2d(in_ch, out_ch,
                              kernel_size=(img_size, 1),
                              stride=1, padding=0)

    def forward(self, x):
        return self.conv(x).squeeze(2)    # (B, row_col_channels, 128)


class PRCCNv2(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.row_extractor = RowExtractor(cfg)
        self.col_extractor = ColExtractor(cfg)

        # after stacking row & col: (B, row_col_channels, 2, img_size)
        in_ch = cfg["row_col_channels"]

        conv2d_blocks = []
        for (out_ch, kh, kw, pad_w, use_pool) in cfg["conv2d_layers"]:
            conv2d_blocks.append(
                nn.Conv2d(in_ch, out_ch,
                          kernel_size=(kh, kw),
                          stride=(1, 1),
                          padding=(0, pad_w))
            )
            conv2d_blocks.append(nn.BatchNorm2d(out_ch))
            conv2d_blocks.append(nn.ReLU())
            if use_pool:
                conv2d_blocks.append(nn.MaxPool2d(kernel_size=(1, 2)))
            in_ch = out_ch

        self.conv2d_blocks = nn.Sequential(*conv2d_blocks)
        self.global_pool   = nn.AdaptiveAvgPool2d((1, 4))
        flat_size          = in_ch*4  # last out_ch from conv2d_layers

        fc_blocks = []
        prev = flat_size
        for hidden in cfg["fc_layers"]:
            fc_blocks += [nn.Linear(prev, hidden), nn.ReLU(), nn.Dropout(cfg["dropout"])]
            prev = hidden
        fc_blocks.append(nn.Linear(prev, cfg["num_classes"]))
        self.fc = nn.Sequential(*fc_blocks)

    def forward(self, x):
        row_out = self.row_extractor(x).unsqueeze(2)    # (B, C, 1, 128)
        col_out = self.col_extractor(x).unsqueeze(2)    # (B, C, 1, 128)
        out     = torch.cat([row_out, col_out], dim=2)  # (B, C, 2, 128)
        out     = self.conv2d_blocks(out)
        out     = self.global_pool(out).flatten(1)
        return self.fc(out)
    
def run_epoch(model, loader, criterion, optimizer, device, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_probs, all_labels = [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs   = imgs.to(device)
            labels = labels.squeeze().long().to(device)
            if train:
                optimizer.zero_grad()
            out  = model(imgs)
            loss = criterion(out, labels)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            correct    += (out.argmax(1) == labels).sum().item()
            total      += imgs.size(0)
            all_probs.append(torch.softmax(out, dim=1).cpu().detach().numpy())
            all_labels.append(labels.cpu().detach().numpy())
    all_probs  = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)
    auc = roc_auc_score(all_labels, all_probs, multi_class="ovr", average="macro")
    return total_loss / total, correct / total, auc


train_ds = raw_train
val_ds   = raw_val

# ─── COMPUTE DATASET MEAN & STD ──────────────────────────────────────────────
temp_transform = transforms.Compose([
    transforms.Resize((CV_CONFIG["image_size"], CV_CONFIG["image_size"])),
    transforms.Grayscale(num_output_channels=CV_CONFIG["in_channels"]),
    transforms.ToTensor(),
])

train_ds.transform = temp_transform
loader = DataLoader(train_ds, batch_size=256, shuffle=False)

mean = 0.0
std  = 0.0
nb_samples = 0

for images, _ in loader:
    batch_size = images.size(0)
    images     = images.view(batch_size, images.size(1), -1)
    mean      += images.mean(2).sum(0)
    std       += images.std(2).sum(0)
    nb_samples += batch_size

mean /= nb_samples
std  /= nb_samples
print(f"Computed Mean : {mean}")
print(f"Computed Std  : {std}")

# ─── FINAL TRANSFORM ─────────────────────────────────────────────────────────
transform = transforms.Compose([
    transforms.Resize((CV_CONFIG["image_size"], CV_CONFIG["image_size"])),
    transforms.Grayscale(num_output_channels=CV_CONFIG["in_channels"]),
    transforms.ToTensor(),
    transforms.Normalize(mean.tolist(), std.tolist()),
])

train_ds.transform = transform
val_ds.transform   = transform
print("Normalization attached successfully.")

# ─── 5-FOLD CV ────────────────────────────────────────────────────────────────
full_dataset = torch.utils.data.ConcatDataset([train_ds, val_ds])
all_labels   = np.array(
    [train_ds.targets[i] for i in range(len(train_ds))] +
    [val_ds.targets[i]   for i in range(len(val_ds))]
)

skf = StratifiedKFold(n_splits=CV_CONFIG["n_folds"], shuffle=True, random_state=42)
fold_results = []
total_params = None

print("="*65)
print("PRCCNv2 — 5-Fold Cross Validation — KMNIST")
print("="*65)

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(all_labels)), all_labels)):
    print(f"\n── Fold {fold+1}/{CV_CONFIG['n_folds']} ──────────────────────────────────")

    fold_train = Subset(full_dataset, train_idx)
    fold_val   = Subset(full_dataset, val_idx)

    fold_train_loader = DataLoader(fold_train, batch_size=CV_CONFIG["batch_size"], shuffle=True,  num_workers=2)
    fold_val_loader   = DataLoader(fold_val,   batch_size=CV_CONFIG["batch_size"], shuffle=False, num_workers=2)

    model     = PRCCNv2(CV_CONFIG).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=CV_CONFIG["lr"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

    if total_params is None:
        total_params = sum(p.numel() for p in model.parameters())
        print(f"Total params: {total_params:,}")

    best_fold_auc = 0.0

    for epoch in range(1, CV_CONFIG["epochs"] + 1):
        tr_loss, tr_acc, tr_auc = run_epoch(model, fold_train_loader, criterion, optimizer, device, train=True)
        vl_loss, vl_acc, vl_auc = run_epoch(model, fold_val_loader,   criterion, None,      device, train=False)
        scheduler.step(vl_loss)

        print(f"  Epoch {epoch:02d}/{CV_CONFIG['epochs']}  "
              f"tr_loss: {tr_loss:.4f}  tr_acc: {tr_acc:.4f}  "
              f"vl_loss: {vl_loss:.4f}  vl_acc: {vl_acc:.4f}  vl_auc: {vl_auc:.4f}")

        if vl_auc > best_fold_auc:
            best_fold_auc = vl_auc

    fold_results.append(best_fold_auc)
    print(f"  Fold {fold+1} best AUC: {best_fold_auc:.4f}")

# ─── SUMMARY ──────────────────────────────────────────────────────────────────
fold_results = np.array(fold_results)
print("\n" + "="*65)
print("PRCCNv2 CV SUMMARY")
print("="*65)
print(f"Fold AUCs : {[f'{x:.4f}' for x in fold_results]}")
print(f"Mean AUC  : {fold_results.mean():.4f}")
print(f"Std AUC   : {fold_results.std():.4f}")
print(f"Params    : {total_params:,}")

Computed Mean : tensor([0.2861])
Computed Std  : tensor([0.3042])
Normalization attached successfully.
PRCCNv2 — 5-Fold Cross Validation — KMNIST

── Fold 1/5 ──────────────────────────────────
Total params: 71,050
  Epoch 01/20  tr_loss: 0.5947  tr_acc: 0.7881  vl_loss: 0.4303  vl_acc: 0.8400  vl_auc: 0.9863
  Epoch 02/20  tr_loss: 0.4135  tr_acc: 0.8526  vl_loss: 0.3790  vl_acc: 0.8619  vl_auc: 0.9886
  Epoch 03/20  tr_loss: 0.3746  tr_acc: 0.8646  vl_loss: 0.3422  vl_acc: 0.8736  vl_auc: 0.9901
  Epoch 04/20  tr_loss: 0.3473  tr_acc: 0.8728  vl_loss: 0.3307  vl_acc: 0.8793  vl_auc: 0.9905
  Epoch 05/20  tr_loss: 0.3302  tr_acc: 0.8806  vl_loss: 0.3303  vl_acc: 0.8748  vl_auc: 0.9909
  Epoch 06/20  tr_loss: 0.3170  tr_acc: 0.8839  vl_loss: 0.3223  vl_acc: 0.8804  vl_auc: 0.9915
  Epoch 07/20  tr_loss: 0.3028  tr_acc: 0.8877  vl_loss: 0.3141  vl_acc: 0.8851  vl_auc: 0.9918
  Epoch 08/20  tr_loss: 0.2925  tr_acc: 0.8929  vl_loss: 0.3213  vl_acc: 0.8824  vl_auc: 0.9914
  Epoch 09/20  tr

In [5]:
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import Subset
from torchmetrics.classification import MulticlassAUROC
# ─── BASELINE CNN CONFIG ──────────────────────────────────────────────────────
BASELINE_CONFIG = {
    "image_size"       : 128,
    "in_channels"      : 1,
    "num_classes"      : 10,
    "conv2d_layers"    : [
        # (out_ch, kh, kw, pad, use_pool)
        (16,  3, 3, 1, False),
        (32,  3, 3, 1, True),
        (64,  3, 3, 1, True),
        (128, 3, 3, 1, True),
    ],
    "global_pool_size" : 4,
    "fc_layers"        : [64, 32],
    "dropout"          : 0.1,
    "batch_size"       : 128,
    "epochs"           : 20,
    "lr"               : 1e-3,
    "n_folds"          : 3,
    "num_workers"      : 4,
    "pin_memory"       : True,
}
# ─────────────────────────────────────────────────────────────────────────────


class BaselineCNN(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        in_ch = cfg["in_channels"]
        conv2d_blocks = []

        for (out_ch, kh, kw, pad, use_pool) in cfg["conv2d_layers"]:
            conv2d_blocks += [
                nn.Conv2d(in_ch, out_ch, kernel_size=(kh, kw),
                          stride=(1, 1), padding=(pad, pad)),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
            ]
            if use_pool:
                conv2d_blocks.append(nn.MaxPool2d(kernel_size=(2, 2)))
            in_ch = out_ch

        self.conv2d_blocks = nn.Sequential(*conv2d_blocks)

        gp = cfg["global_pool_size"]
        self.global_pool = nn.AdaptiveAvgPool2d((gp, gp))
        flat_size        = in_ch * gp * gp

        fc_blocks = []
        prev = flat_size
        for hidden in cfg["fc_layers"]:
            fc_blocks += [nn.Linear(prev, hidden), nn.ReLU(inplace=True),
                          nn.Dropout(cfg["dropout"])]
            prev = hidden
        fc_blocks.append(nn.Linear(prev, cfg["num_classes"]))
        self.fc = nn.Sequential(*fc_blocks)

    def forward(self, x):
        out = self.conv2d_blocks(x)
        out = self.global_pool(out).flatten(1)
        return self.fc(out)


def run_epoch(model, loader, criterion, optimizer, device, auc_metric, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0

    auc_metric.reset()

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs   = imgs.to(device, non_blocking=True)
            labels = labels.squeeze().long().to(device, non_blocking=True)

            if train:
                optimizer.zero_grad(set_to_none=True)

            out  = model(imgs)
            loss = criterion(out, labels)

            if train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * imgs.size(0)
            correct    += (out.argmax(1) == labels).sum().item()
            total      += imgs.size(0)
            auc_metric.update(out.detach(), labels)

    auc = auc_metric.compute().item()
    return total_loss / total, correct / total, auc


train_ds = raw_train
val_ds   = raw_val

# ─── COMPUTE DATASET MEAN & STD ──────────────────────────────────────────────
temp_transform = transforms.Compose([
    transforms.Resize((BASELINE_CONFIG["image_size"], BASELINE_CONFIG["image_size"])),
    transforms.Grayscale(num_output_channels=BASELINE_CONFIG["in_channels"]),
    transforms.ToTensor(),
])
train_ds.transform = temp_transform

loader = DataLoader(train_ds, batch_size=512, shuffle=False,
                    num_workers=BASELINE_CONFIG["num_workers"],
                    pin_memory=BASELINE_CONFIG["pin_memory"])

mean, std, nb_samples = 0.0, 0.0, 0
for images, _ in loader:
    bs     = images.size(0)
    images = images.view(bs, images.size(1), -1)
    mean  += images.mean(2).sum(0)
    std   += images.std(2).sum(0)
    nb_samples += bs

mean /= nb_samples
std  /= nb_samples
print(f"Computed Mean : {mean}  Std : {std}")

# ─── FINAL TRANSFORM ─────────────────────────────────────────────────────────
transform = transforms.Compose([
    transforms.Resize((BASELINE_CONFIG["image_size"], BASELINE_CONFIG["image_size"])),
    transforms.Grayscale(num_output_channels=BASELINE_CONFIG["in_channels"]),
    transforms.ToTensor(),
    transforms.Normalize(mean.tolist(), std.tolist()),
])
train_ds.transform = transform
val_ds.transform   = transform
print("Normalization attached.")

# ─── 5-FOLD CV ────────────────────────────────────────────────────────────────
full_dataset = torch.utils.data.ConcatDataset([train_ds, val_ds])
all_labels   = np.array(
    [train_ds.targets[i] for i in range(len(train_ds))] +
    [val_ds.targets[i]   for i in range(len(val_ds))]
)

skf          = StratifiedKFold(n_splits=BASELINE_CONFIG["n_folds"], shuffle=True, random_state=42)
fold_results = []
total_params = None

auc_metric = MulticlassAUROC(num_classes=BASELINE_CONFIG["num_classes"],
                              average="macro").to(device)

print("="*65)
print("BaselineCNN — 5-Fold Cross Validation — KMNIST")
print("="*65)

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(all_labels)), all_labels)):
    print(f"\n── Fold {fold+1}/{BASELINE_CONFIG['n_folds']} ──────────────────────────────")

    fold_train = Subset(full_dataset, train_idx)
    fold_val   = Subset(full_dataset, val_idx)

    fold_train_loader = DataLoader(
        fold_train, batch_size=BASELINE_CONFIG["batch_size"], shuffle=True,
        num_workers=BASELINE_CONFIG["num_workers"], pin_memory=BASELINE_CONFIG["pin_memory"],
        persistent_workers=True, prefetch_factor=2,
    )
    fold_val_loader = DataLoader(
        fold_val, batch_size=BASELINE_CONFIG["batch_size"] * 2, shuffle=False,
        num_workers=BASELINE_CONFIG["num_workers"], pin_memory=BASELINE_CONFIG["pin_memory"],
        persistent_workers=True, prefetch_factor=2,
    )

    model     = BaselineCNN(BASELINE_CONFIG).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=BASELINE_CONFIG["lr"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

    if total_params is None:
        total_params = sum(p.numel() for p in model.parameters())
        print(f"Total params: {total_params:,}")

    best_fold_auc = 0.0

    for epoch in range(1, BASELINE_CONFIG["epochs"] + 1):
        tr_loss, tr_acc, tr_auc = run_epoch(model, fold_train_loader, criterion,
                                            optimizer, device, auc_metric, train=True)
        vl_loss, vl_acc, vl_auc = run_epoch(model, fold_val_loader,   criterion,
                                            None,      device, auc_metric, train=False)
        scheduler.step(vl_loss)

        print(f"  Epoch {epoch:02d}/{BASELINE_CONFIG['epochs']}  "
              f"tr_loss:{tr_loss:.4f}  tr_acc:{tr_acc:.4f}  "
              f"vl_loss:{vl_loss:.4f}  vl_acc:{vl_acc:.4f}  vl_auc:{vl_auc:.4f}")

        if vl_auc > best_fold_auc:
            best_fold_auc = vl_auc

    fold_results.append(best_fold_auc)
    print(f"  Fold {fold+1} best AUC: {best_fold_auc:.4f}")

# ─── SUMMARY ──────────────────────────────────────────────────────────────────
fold_results = np.array(fold_results)
print("\n" + "="*65)
print("BaselineCNN CV SUMMARY")
print("="*65)
print(f"Fold AUCs : {[f'{x:.4f}' for x in fold_results]}")
print(f"Mean AUC  : {fold_results.mean():.4f}")
print(f"Std AUC   : {fold_results.std():.4f}")
print(f"Params    : {total_params:,}")

Computed Mean : tensor([0.2861])  Std : tensor([0.3042])
Normalization attached.
BaselineCNN — 5-Fold Cross Validation — KMNIST

── Fold 1/3 ──────────────────────────────
Total params: 231,178
  Epoch 01/20  tr_loss:0.6455  tr_acc:0.7630  vl_loss:0.4693  vl_acc:0.8143  vl_auc:0.9840
  Epoch 02/20  tr_loss:0.3939  tr_acc:0.8571  vl_loss:0.3223  vl_acc:0.8832  vl_auc:0.9910
  Epoch 03/20  tr_loss:0.3363  tr_acc:0.8808  vl_loss:0.2806  vl_acc:0.8992  vl_auc:0.9930
  Epoch 04/20  tr_loss:0.2982  tr_acc:0.8933  vl_loss:0.2686  vl_acc:0.9032  vl_auc:0.9936
  Epoch 05/20  tr_loss:0.2787  tr_acc:0.9004  vl_loss:0.2795  vl_acc:0.8992  vl_auc:0.9940
  Epoch 06/20  tr_loss:0.2617  tr_acc:0.9074  vl_loss:0.2865  vl_acc:0.8933  vl_auc:0.9939
  Epoch 07/20  tr_loss:0.2488  tr_acc:0.9109  vl_loss:0.2938  vl_acc:0.8912  vl_auc:0.9937
  Epoch 08/20  tr_loss:0.2390  tr_acc:0.9137  vl_loss:0.2434  vl_acc:0.9112  vl_auc:0.9948
  Epoch 09/20  tr_loss:0.2319  tr_acc:0.9160  vl_loss:0.2384  vl_acc:0.9148  v